# LoRA Finetuning

Finetunes a LoRA adapter on StableDiffusion 1.5's UNet.


Runtime: Colab A100


Dataset: Persian miniature paintings (Wikimedia)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/HarshaSrirangam/stable-diffusion-rebuilt/blob/main/notebooks/train.ipynb)

## Environment/Setup

In [1]:
import torch
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device) # should be cuda

cuda


In [13]:
# clone repo
!git clone https://github.com/HarshaSrirangam/stable-diffusion-rebuilt.git
%cd stable-diffusion-rebuilt
!git pull

Cloning into 'stable-diffusion-rebuilt'...
remote: Enumerating objects: 480, done.
remote: Counting objects: 100% (4/4), done.
remote: Compressing objects: 100% (4/4), done.
remote: Total 480 (delta 0), reused 1 (delta 0), pack-reused 476 (from 1)
Receiving objects: 100% (480/480), 4.01 MiB | 19.00 MiB/s, done.
Resolving deltas: 100% (255/255), done.
/content/stable-diffusion-rebuilt/stable-diffusion-rebuilt
Already up to date.


In [3]:
# install dependencies (skip torch/numpy to avoid overwriting Colab's preinstalled packages)
%pip install -e . --no-deps -q
%pip install transformers safetensors tqdm accelerate pyyaml datasets -q

  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for stable-diffusion-rebuilt (pyproject.toml) ... done


In [4]:
# mount drive
from google.colab import drive
drive.mount("/content/drive", force_remount=True)

Mounted at /content/drive


In [20]:
# symlink runs/ and data/ (weights, cache, persian)
!ln -sfn /content/drive/MyDrive/sd-rebuilt/data data
!ln -sfn /content/drive/MyDrive/sd-rebuilt/runs runs

!ls runs data

data:
cache  persian	weights

runs:
naruto_r16_selfcross_lora	persian_r16_selfcross_lora
naruto_r16_selfcross_lora_test


## Training

These configs are written to `configs/lora.yaml`. The `runs/<run_name>` for this run comes from this config. If a run already exists, the `name` field in `config/lora.yaml` should be modified.

In [8]:
%%writefile configs/lora.yaml

name: lora
pretrained_path: data/weights/v1-5-pruned-emaonly.safetensors
device: cuda

dataset: persian

r: 16
alpha: 16
targets:
  desc: selfcross
  layers: ["q1", "k1", "v1", "proj_out1", "q2", "k2", "v2", "proj_out2"]

n_epochs: 10
batch_size: 8
lr: 0.0001
seed: 42
log_interval: 10

Overwriting configs/lora.yaml


In [9]:
# training
!python scripts/train_lora.py


>>> Loading UNet and injecting LoRA layers

>>> Preparing dataset

>>> Precomputing image/caption embeddings (no cache found)
Encoding image/caption batches: 100% 163/163 [17:40<00:00,  6.51s/it]

>>> Training
epoch 1/10: 100% 163/163 [00:35<00:00,  4.65it/s]
epoch 2/10: 100% 163/163 [00:33<00:00,  4.83it/s]
epoch 3/10: 100% 163/163 [00:33<00:00,  4.82it/s]
epoch 4/10: 100% 163/163 [00:33<00:00,  4.83it/s]
epoch 5/10: 100% 163/163 [00:33<00:00,  4.84it/s]
epoch 6/10: 100% 163/163 [00:33<00:00,  4.84it/s]
epoch 7/10: 100% 163/163 [00:33<00:00,  4.81it/s]
epoch 8/10: 100% 163/163 [00:33<00:00,  4.81it/s]
epoch 9/10: 100% 163/163 [00:33<00:00,  4.81it/s]
epoch 10/10: 100% 163/163 [00:33<00:00,  4.81it/s]

>>> Done


In [16]:
import yaml
cfg = yaml.safe_load(open("configs/lora.yaml"))
run_name = f"{cfg['dataset']}_r{cfg['r']}_{cfg['targets']['desc']}_{cfg['name']}"
print(run_name) # must match run directory created in previous cell

persian_r16_selfcross_lora


In [22]:
# eval
!python scripts/evaluate_lora.py --run {run_name}


>>> Loading model

>>> Loading LoRA adapter

>>> Generating eval images
Denoising: 100% 50/50 [00:04<00:00, 11.87it/s]
Denoising: 100% 50/50 [00:03<00:00, 12.64it/s]
Denoising: 100% 50/50 [00:03<00:00, 12.65it/s]
Denoising: 100% 50/50 [00:03<00:00, 12.66it/s]
Denoising: 100% 50/50 [00:03<00:00, 12.65it/s]
Denoising: 100% 50/50 [00:03<00:00, 12.66it/s]
Denoising: 100% 50/50 [00:03<00:00, 12.66it/s]
Denoising: 100% 50/50 [00:03<00:00, 12.68it/s]
Denoising: 100% 50/50 [00:03<00:00, 12.68it/s]
Denoising: 100% 50/50 [00:03<00:00, 12.67it/s]

>>> Computing CLIP prompt adherence

>>> Computing CLIP style adherence

>>> Running UNet on eval dataset

>>> Precomputing image/caption embeddings (no cache found)
Encoding image/caption batches: 100% 21/21 [00:08<00:00,  2.40it/s]

>>> Done. Eval outputs written to /content/stable-diffusion-rebuilt/stable-diffusion-rebuilt/runs/persian_r16_selfcross_lora/eval


## Outputs
- Checkpoints: `runs/<run_name>/checkpoints/checkpoint-<epoch>.pt`
- Loss curve: `runs/<run_name>/losses.json`
- Frozen config: `runs/<run_name>/training_config.yaml`
- Eval files: `runs/<run_name>/eval/`